# Model Analysis

In [ ]:
import sys
sys.path.insert(0, '..')
import json
import pickle
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Load metrics
with open('../output/metrics.json') as f:
    metrics = json.load(f)

for m in metrics:
    print(f"{m['model']:20s} | LogLoss: {m['log_loss']:.4f} | AUC: {m['roc_auc']:.4f} | Top1: {m['top_1_accuracy']:.1%} | Top3: {m['top_3_accuracy']:.1%} | Top5: {m['top_5_accuracy']:.1%}")

In [ ]:
# Compare models visually
test_metrics = [m for m in metrics if 'test' in m['model']]
names = [m['model'] for m in test_metrics]
top1 = [m['top_1_accuracy'] for m in test_metrics]
top3 = [m['top_3_accuracy'] for m in test_metrics]
top5 = [m['top_5_accuracy'] for m in test_metrics]

x = np.arange(len(names))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, top1, width, label='Top-1')
ax.bar(x, top3, width, label='Top-3')
ax.bar(x + width, top5, width, label='Top-5')
ax.set_ylabel('Accuracy')
ax.set_title('Top-K Accuracy by Model (Test Set)')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from LightGBM
from PIL import Image
img = Image.open('../output/plots/importance_LightGBM.png')
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Error analysis: which positions get predicted vs actual
features = pl.read_parquet('../data/processed/features.parquet')
with open('../models/lgbm.pkl', 'rb') as f:
    lgbm = pickle.load(f)
with open('../models/feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

test = features.filter(pl.col('season') == 2024)
X = test.select(feature_cols).to_numpy().astype(float)
X = np.nan_to_num(X, nan=0.0)
probs = lgbm.predict_proba(X)[:, 1]
test = test.with_columns(pl.Series('prob', probs))

# Top-1 predicted per game
top1_pred = test.sort(['game_id', 'prob'], descending=[False, True]).group_by('game_id').first()
pos_cols = [c for c in top1_pred.columns if c.startswith('position_')]
if pos_cols:
    print('Position distribution of top-1 predicted:')
    for col in pos_cols:
        pct = top1_pred[col].sum() / top1_pred.shape[0]
        print(f'  {col}: {pct:.1%}')